# 02 — Kartın mikrofonu ve senin sesinle yeniden eğitim

**Amaç:** Modeli, kartın mikrofonundan alınmış kendi kayıtlarınla da eğitip senin sesindeki doğruluğu artırmak.

**Kural:** Normalizasyon değerleri (`mu`, `sd`) **kartta kullanılanlarla aynı kalır** (`mfcc_consts.h`'den okunur). Böylece kartta sadece model dosyası ve iki sayı (giriş ölçeği) değişir.

**Dürüst test:** Her kelimenin son 3 kaydı eğitime **hiç girmez**; eski ve yeni model onlarla karşılaştırılır.

Hücreleri yukarıdan aşağı sırayla çalıştır.

## 1. Dosyaları yükle
`kws_egitim.zip`, `mfcc_consts.h` ve eski `kws_int8.tflite` dosyalarının üçünü birden seç.

In [ ]:
import pathlib, re, zipfile
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from scipy.io import wavfile
from sklearn.model_selection import train_test_split
from google.colab import files

up = files.upload()
with zipfile.ZipFile('kws_egitim.zip') as z:
    z.extractall('.')
print(sorted(p.name for p in pathlib.Path('kws_egitim').iterdir() if p.is_dir()))

## 2. Kartta kullanılan sabitleri oku (mu, sd, giriş ölçeği)

In [ ]:
h = open('mfcc_consts.h').read()
def c_arr(name):
    body = re.search(r'static const float ' + name + r'\[\d+\] = \{(.*?)\};', h, re.S).group(1)
    return np.array(re.findall(r'(-?\d\.\d+e[+-]\d+)f', body), dtype=np.float32)
mu, sd = c_arr('MFCC_MU'), c_arr('MFCC_SD')
old_scale = float(re.search(r'#define MODEL_IN_SCALE\s+([-\d.e+]+)f', h).group(1))
old_zp = int(re.search(r'#define MODEL_IN_ZP\s+\((-?\d+)\)', h).group(1))
print('mu =', mu.round(2)); print('sd =', sd.round(2)); print('eski giris olcegi:', old_scale, old_zp)

## 3. Veri seti ve MFCC (Colab'daki ve karttaki ile birebir aynı)

In [ ]:
!wget -q -nc http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip
!unzip -qo mini_speech_commands.zip -d data
data_dir = pathlib.Path('data/mini_speech_commands')
labels = sorted(p.name for p in data_dir.iterdir() if p.is_dir())
print(labels)

def load_wav(path, target_len=16000):
    _, x = wavfile.read(path)
    x = x.astype(np.float32) / 32768.0
    if len(x) < target_len:
        x = np.pad(x, (0, target_len - len(x)))
    return x[:target_len]

def mfcc(x, sr=16000):
    stft = tf.signal.stft(x, frame_length=640, frame_step=320, fft_length=1024)
    spec = tf.abs(stft)
    mel_w = tf.signal.linear_to_mel_weight_matrix(40, 513, sr, 20.0, 4000.0)
    mel = tf.tensordot(spec, mel_w, 1)
    return tf.signal.mfccs_from_log_mel_spectrograms(tf.math.log(mel + 1e-6))[:, :10].numpy()

norm = lambda a: (a - mu) / sd

## 4. Orijinal veri setinin MFCC'si (birkaç dakika sürer)

In [ ]:
X, y = [], []
for i, lab in enumerate(labels):
    for f in sorted((data_dir / lab).glob('*.wav')):
        X.append(mfcc(load_wav(f))); y.append(i)
X, y = np.array(X, np.float32), np.array(y)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)
X_tr, X_val, X_te = norm(X_tr), norm(X_val), norm(X_te)
print(X_tr.shape, X_val.shape, X_te.shape)

## 5. Senin kayıtların: eğitim / test ayrımı ve çoğaltma (augmentation)
Her kelimenin **son 3 kaydı test**, geri kalanı eğitim. Eğitim kayıtlarının her birinden 20 çeşitleme üretiyoruz: zamanda kaydırma (±125 ms), ses seviyesi (×0,6–1,4) ve hafif gürültü.

In [ ]:
rng = np.random.default_rng(0)
def augment(x):
    s = int(rng.integers(-2000, 2001))
    z = np.roll(x, s)
    if s > 0: z[:s] = 0
    elif s < 0: z[s:] = 0
    z = z * rng.uniform(0.6, 1.4)
    rms = np.sqrt(np.mean(z ** 2)) + 1e-9
    z = z + rng.normal(0, rms / 10 ** (rng.uniform(15, 40) / 20), z.shape)
    return np.clip(z, -1, 1).astype(np.float32)

N_TEST, N_AUG = 3, 20
Xm_tr, ym_tr, Xm_te, ym_te = [], [], [], []
for i, lab in enumerate(labels):
    fs = sorted(pathlib.Path('kws_egitim', lab).glob('*.wav'))
    print(f'{lab:6s}: {len(fs)} kayit')
    for f in fs[:-N_TEST]:
        x = load_wav(f)
        Xm_tr.append(mfcc(x)); ym_tr.append(i)
        for _ in range(N_AUG):
            Xm_tr.append(mfcc(augment(x))); ym_tr.append(i)
    for f in fs[-N_TEST:]:
        Xm_te.append(mfcc(load_wav(f))); ym_te.append(i)
Xm_tr, ym_tr = norm(np.array(Xm_tr, np.float32)), np.array(ym_tr)
Xm_te, ym_te = norm(np.array(Xm_te, np.float32)), np.array(ym_te)
print('senin egitim:', Xm_tr.shape, ' senin test:', Xm_te.shape)

## 6. Eski modelin senin test kayıtlarındaki başarısı (karşılaştırma için)

In [ ]:
def eval_tflite(model_bytes, Xn, yy):
    it = tf.lite.Interpreter(model_content=model_bytes); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    s, zp = i['quantization']
    pred = []
    for k in range(len(Xn)):
        q = np.clip(np.round(Xn[k:k+1, ..., None] / s + zp), -128, 127).astype(np.int8)
        it.set_tensor(i['index'], q); it.invoke()
        pred.append(int(np.argmax(it.get_tensor(o['index'])[0])))
    pred = np.array(pred)
    return (pred == yy).mean(), pred

old_model = open('kws_int8.tflite', 'rb').read()
acc_old_me, pred_old = eval_tflite(old_model, Xm_te, ym_te)
acc_old_ds, _ = eval_tflite(old_model, X_te, y_te)
print(f'ESKI model -> senin sesin: %{100*acc_old_me:.0f}   veri seti testi: %{100*acc_old_ds:.1f}')
for k in range(len(ym_te)):
    print(f'  {labels[ym_te[k]]:6s} -> {labels[pred_old[k]]}')

## 7. Yeniden eğitim (aynı mimari, orijinal veri + senin kayıtların)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(1)

model = keras.Sequential([
    layers.Input(shape=(49, 10, 1)),
    layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(len(labels), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

X_all = np.concatenate([X_tr, Xm_tr])[..., None]
y_all = np.concatenate([y_tr, ym_tr])
hist = model.fit(X_all, y_all, validation_data=(X_val[..., None], y_val),
                 epochs=40, batch_size=64, shuffle=True,
                 callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

## 8. int8'e çevir ve iki modeli karşılaştır

In [ ]:
rep = np.concatenate([X_tr[:300], Xm_tr[:100]])
def rep_data():
    for k in range(len(rep)):
        yield [rep[k:k+1, ..., None].astype(np.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
new_model = conv.convert()

acc_new_me, pred_new = eval_tflite(new_model, Xm_te, ym_te)
acc_new_ds, _ = eval_tflite(new_model, X_te, y_te)
print(f'ESKI model -> senin sesin: %{100*acc_old_me:.0f}   veri seti testi: %{100*acc_old_ds:.1f}')
print(f'YENI model -> senin sesin: %{100*acc_new_me:.0f}   veri seti testi: %{100*acc_new_ds:.1f}')
for k in range(len(ym_te)):
    print(f'  {labels[ym_te[k]]:6s} -> eski: {labels[pred_old[k]]:6s} yeni: {labels[pred_new[k]]}')

## 9. Yeni model ve güncellenmiş `mfcc_consts.h`'yi indir

In [ ]:
it = tf.lite.Interpreter(model_content=new_model); it.allocate_tensors()
s_new, zp_new = it.get_input_details()[0]['quantization']
print('yeni giris olcegi:', s_new, zp_new, '  (eski:', old_scale, old_zp, ')')

open('kws_int8_v2.tflite', 'wb').write(new_model)
h2 = re.sub(r'#define MODEL_IN_SCALE\s+[-\d.e+]+f', f'#define MODEL_IN_SCALE  {s_new:.10e}f', h)
h2 = re.sub(r'#define MODEL_IN_ZP\s+\(-?\d+\)', f'#define MODEL_IN_ZP     ({zp_new})', h2)
open('mfcc_consts.h', 'w').write(h2)
files.download('kws_int8_v2.tflite')
files.download('mfcc_consts.h')